In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Imports loaded.")
DB_HOST = "host.docker.internal"
DB_PORT = 5432
DB_NAME = "studybook"
DB_USER = "sb_user"
DB_PASSWORD = "sb_pass_123"

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("Database URL created.")

Imports loaded.
Database URL created.


# Cell 5 — Helper function to run SQL

In [2]:
def run_sql(sql: str) -> pd.DataFrame:
    """
    Run SQL against the local PostgreSQL telemetry lab
    and return the result as a pandas DataFrame.
    """
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)

In [3]:
# ============================================================
# Cell 1: Database Identity Smoke Test
# Purpose:
# Confirm Python is connected to the expected PostgreSQL database.
# ============================================================

query = """
SELECT
    current_database() AS database_name
    ,current_user AS connected_user
    ,inet_server_addr() AS server_ip
    ,inet_server_port() AS server_port
    ,now() AS database_time
    ;
"""

df_identity = pd.read_sql(query, engine)

df_identity

,database_name,connected_user,server_ip,server_port,database_time
0,studybook,sb_user,172.18.0.2,5432,2026-05-07 23:51:20.111820+00:00


In [4]:
# ============================================================
# Cell 1: Database Identity Smoke Test
# Purpose:
# Confirm Python is connected to the expected PostgreSQL database.
# ============================================================

query = """
SELECT
    current_user AS connected_user
    ,session_user AS suser
    ,current_schema() AS my_schema
    ;
"""

df_identity2 = pd.read_sql(query, engine)

df_identity2

,connected_user,suser,my_schema
0,sb_user,sb_user,public


In [5]:
# ============================================================
# Cell 2: List Tables in the Database
# Purpose:
# See what schemas and tables exist in PostgreSQL.
# ============================================================

query = """
SELECT
    table_schema,
    table_name,
    table_type
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY
    table_schema,
    table_name;
"""

df_tables = pd.read_sql(query, engine)

df_tables

,table_schema,table_name,table_type
0,public,capacity_thresholds,BASE TABLE
1,public,deployments,BASE TABLE
2,public,hello,BASE TABLE
3,public,hosts,BASE TABLE
4,public,incidents,BASE TABLE
5,public,server_inventory_practice,BASE TABLE
6,public,server_metric_samples_5min,BASE TABLE
7,public,services,BASE TABLE
8,public,telemetry_samples,BASE TABLE


In [6]:
# ============================================================
# Cell 4: Helper Function - Inspect One Table
# Purpose:
# Reuse this function to inspect columns for any table.
# ============================================================

def inspect_table(table_name, schema_name="public"):
    """
    Show column metadata for one PostgreSQL table.

    Parameters:
        table_name:  name of the table to inspect
        schema_name: schema where the table lives, default is public
    """

    query = f"""
    SELECT
        table_schema,
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = '{schema_name}'
      AND table_name = '{table_name}'
    ORDER BY
        ordinal_position;
    """
    display( run_sql(query))
inspect_table('hello')

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,public,hello,1,id,integer,NO,nextval('hello_id_seq'::regclass)
1,public,hello,2,note,text,NO,None
2,public,hello,3,created_at,timestamp with time zone,YES,now()


In [7]:
# ============================================================
# Cell 4B: Safer Helper Function - Inspect One Table
# Purpose:
# Same result as above, but uses SQL parameters instead of f-string injection.
# ============================================================

def inspect_table_safe(table_name, schema_name="public"):
    """
    Show column metadata for one PostgreSQL table using safe SQL parameters.
    """

    query = """
    SELECT
        table_schema,
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = :schema_name
      AND table_name = :table_name
    ORDER BY
        ordinal_position;
    """

    df = pd.read_sql(
        text(query),
        engine,
        params={
            "schema_name": schema_name,
            "table_name": table_name,
        },
    )

    display(df)
inspect_table_safe('hello')

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,public,hello,1,id,integer,NO,nextval('hello_id_seq'::regclass)
1,public,hello,2,note,text,NO,None
2,public,hello,3,created_at,timestamp with time zone,YES,now()


In [8]:
# ============================================================
# Cell 3: Inspect Table Columns
# Purpose:
# See the columns, data types, and nullable rules for each table.
# ============================================================

query = """
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable,
    column_default
FROM information_schema.columns
WHERE table_schema NOT IN ('pg_catalog', 'information_schema') AND table_name = 'hello'
ORDER BY
    table_schema,
    table_name,
    ordinal_position;
"""

df_columns = run_sql(query)

df_columns

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,public,hello,1,id,integer,NO,nextval('hello_id_seq'::regclass)
1,public,hello,2,note,text,NO,None
2,public,hello,3,created_at,timestamp with time zone,YES,now()


In [9]:
inspect_table('hello')

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,public,hello,1,id,integer,NO,nextval('hello_id_seq'::regclass)
1,public,hello,2,note,text,NO,None
2,public,hello,3,created_at,timestamp with time zone,YES,now()


In [10]:

sql = """
SELECT
    current_user AS connected_user
    ,session_user AS suser
    ,current_schema() AS my_schema
    ;
"""

print("Before:", pd.get_option("display.max_colwidth"))

with pd.option_context(
    "display.max_colwidth", None,
    "display.width", 300
):
    print("Inside:", pd.get_option("display.max_colwidth"))
    display(run_sql(sql))

print("After:", pd.get_option("display.max_colwidth"))

Before: 50
Inside: None


,connected_user,suser,my_schema
0,sb_user,sb_user,public


After: 50
